In [ ]:
import logging
import sys

# Databricks attaches a handler to the root logger before the notebook runs, which
# makes basicConfig a silent no-op; force=True replaces it so INFO records survive
# and stdout puts them in the cell output instead of only the driver log.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

# Quiet noisy third-party loggers (Databricks Python<->JVM bridge, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)

# Uncomment to see the package's DEBUG traces (audio decode, model free):
# logging.getLogger("drzavni_posao").setLevel(logging.DEBUG)

In [ ]:
import gc

import numpy as np
import torch
import whisperx
from pyannote.audio import Model, Inference
from pydantic import BaseModel

from drzavni_posao import (
    SR,
    Episode,
    Transcript,
    MediaStore,
    Transcriber,
    TranscriberConfig,
    Diarizer,
    DiarizerConfig,
    Embedder,
    EmbedderConfig,
    Gallery,
    CharacterAssigner,
    CharacterAssignerConfig,
)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

media = MediaStore("/Users/luka.erceg/Downloads/DRŽAVNI POSAO [HQ] - Ep42-90", "")
tr = Transcriber(
    TranscriberConfig(
        device="cpu",
        compute_type="int8",
        model_name="tiny",
        language="sr",
        align_model_name="dnikolic/wav2vec2-xlsr-530-serbian-colab",
    )
)
di = Diarizer(DiarizerConfig(device="cpu", hf_token=""))

2026-07-05 10:05:09 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint .venv/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


Loaded WhisperX 'tiny' on cpu
2026-07-05 10:05:12 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-3.1


config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

In [ ]:
ep = media.episode("43")
ep = tr.run(ep)
ep = di.run(ep)

In [ ]:
# Labeled reference clips per character — add more entries per character for a
# richer gallery. start/end are in the UNTRIMMED frame of each episode's audio.
SPEAKER_SAMPLES = {
    "Djordje Cvarkov": [
        {"episode_key": "15", "start": 1 * 60 + 39, "end": 1 * 60 + 48},
        {"episode_key": "41", "start": 5 * 60 + 56, "end": 6 * 60 + 14},
        {"episode_key": "46", "start": 2 * 60 + 11, "end": 2 * 60 + 44},
        {"episode_key": "56", "start": 0 * 60 + 36, "end": 0 * 60 + 42},
        {"episode_key": "56", "start": 1 * 60 + 49, "end": 2 * 60 + 0},
        {"episode_key": "56", "start": 2 * 60 + 5, "end": 2 * 60 + 10},
        {"episode_key": "56", "start": 3 * 60 + 34, "end": 3 * 60 + 42},
        {"episode_key": "56", "start": 4 * 60 + 9, "end": 4 * 60 + 25},
    ],
    "Dragan Torbica": [
        {"episode_key": "15", "start": 4 * 60 + 58, "end": 5 * 60 + 13},
        {"episode_key": "40", "start": 2 * 60 + 30, "end": 2 * 60 + 44},
        {"episode_key": "40", "start": 3 * 60 + 48, "end": 4 * 60 + 2},
        {"episode_key": "41", "start": 2 * 60 + 36, "end": 3 * 60 + 2},
        {"episode_key": "56", "start": 0 * 60 + 43, "end": 1 * 60 + 5},
        {"episode_key": "56", "start": 1 * 60 + 40, "end": 1 * 60 + 48},
        {"episode_key": "56", "start": 3 * 60 + 57, "end": 4 * 60 + 6},
    ],
    "Boskic": [
        {"episode_key": "8", "start": 4 * 60 + 48, "end": 5 * 60 + 2},
        {"episode_key": "40", "start": 4 * 60 + 48, "end": 5 * 60 + 10},
        {"episode_key": "56", "start": 2 * 60 + 16, "end": 2 * 60 + 27},
    ],
}

In [ ]:
# Build the gallery once from the reference clips, then the assigner.
# The same embedder embeds the reference clips and, later, each episode's segments.
emb = Embedder(EmbedderConfig(device="cpu", hf_token=""))
gallery = Gallery.from_clips(SPEAKER_SAMPLES, emb, media)
ca = CharacterAssigner(CharacterAssignerConfig(min_similarity=0.5), emb, gallery)

In [ ]:
# ep already went through tr.run + di.run above, so this takes the per-cluster
# path (pyannote speakers → averaged voiceprint → character name). Without di.run
# it would match each segment independently instead.
ep = ca.run(ep)
print(ep.transcript.render())